# Text branch v2: Transformer pipeline for multimodal emotion detection

Continues `DL_text_detection2.ipynb` (BiLSTM baseline on GoEmotions). This notebook is self-contained.

```
text -> RoBERTa encoder -> mean pooling -> 768-d embedding ----+--> 28-way GoEmotions head -> per-emotion thresholds
                                                               |         |
                                                               |         +--> Ekman-7 mapping (anger, disgust, fear, joy, sadness, surprise, neutral)
                                                               |
                                                               +--> exported for the multimodal fusion model (with audio + video features)
```

**Stages**
1. Data and tokenization (subword tokenizer, dynamic padding)
2. Model: pretrained encoder + mean pooling + multi-label head (also returns the embedding for fusion)
3. Imbalance-aware loss (`bce`, `bce_posweight`, `focal`)
4. Training: AMP, warmup + linear decay, layer-group learning rates, early stopping
5. Threshold tuning on validation only (global and per-emotion with a small-class fallback)
6. Evaluation: 28 emotions and Ekman-7, per-class report, error analysis
7. Saving artifacts and a reusable `TextEmotionPredictor`
8. Fusion-ready feature export and a missing-modality-safe fusion module
9. Reusable experiment runner
10. Ablations: loss, model size, seeds, ensemble
11. Fair BiLSTM baseline comparison
12. Probability calibration
13. MELD dialogue fine-tuning and fusion features
14. Knowledge distillation
15. ONNX export and int8 quantization
16. Final summary

## 0. Setup

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn seaborn

In [ ]:
import os, json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModel, AutoConfig,
    DataCollatorWithPadding, get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    f1_score, precision_score, recall_score, hamming_loss, classification_report
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
class CFG:
    model_name       = "roberta-base"    # "distilroberta-base" is ~2x faster, slightly lower F1
    max_length       = 64                # GoEmotions comments are short; 64 subword tokens covers almost all
    batch_size       = 32
    epochs           = 4
    lr_encoder       = 2e-5
    lr_head          = 1e-3
    weight_decay     = 0.01
    warmup_ratio     = 0.06
    dropout          = 0.1
    loss_type        = "bce_posweight"   # "bce" | "bce_posweight" | "focal"
    pos_weight_clip  = 10.0
    focal_gamma      = 2.0
    select_threshold = 0.3               # fixed threshold used ONLY to pick the best epoch
    patience         = 2
    out_dir          = "artifacts_text"

os.makedirs(CFG.out_dir, exist_ok=True)

## 1. Data and tokenization

In [ ]:
ds = load_dataset("google-research-datasets/go_emotions", "simplified")

label_names = ds["train"].features["labels"].feature.names
NUM_LABELS = len(label_names)
id2label = {i: l for i, l in enumerate(label_names)}
label2id = {l: i for i, l in enumerate(label_names)}

print(ds)
print("Number of emotions:", NUM_LABELS)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)

def multi_hot(labels):
    v = np.zeros(NUM_LABELS, dtype=np.float32)
    v[labels] = 1.0
    return v.tolist()

def preprocess(batch):
    enc = tokenizer(batch["text"], truncation=True, max_length=CFG.max_length)
    enc["labels"] = [multi_hot(l) for l in batch["labels"]]
    return enc

test_texts = list(ds["test"]["text"])      # kept for error analysis
tok_ds = ds.map(preprocess, batched=True, remove_columns=ds["train"].column_names)

lens = [len(x) for x in tok_ds["train"]["input_ids"]]
print("Token length  mean: %.1f | p95: %d | p99: %d | max: %d" % (
    np.mean(lens), np.percentile(lens, 95), np.percentile(lens, 99), max(lens)))
print("Truncated at max_length: %.2f%%" % (100 * np.mean([l >= CFG.max_length for l in lens])))

In [ ]:
collator = DataCollatorWithPadding(tokenizer)   # pads each batch to its own longest sequence

def make_loader(split, shuffle=False, batch_size=None):
    return DataLoader(
        tok_ds[split],
        batch_size=batch_size or CFG.batch_size,
        shuffle=shuffle,
        collate_fn=collator,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

train_loader = make_loader("train", shuffle=True)
val_loader   = make_loader("validation", batch_size=128)
test_loader  = make_loader("test", batch_size=128)

batch = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in batch.items()})

## 2. Model

Mean pooling over non-padding tokens, then dropout and a linear head. `forward(..., return_embedding=True)` also returns the pooled vector, which is what the multimodal fusion model consumes.

In [ ]:
class EmotionTransformer(nn.Module):
    def __init__(self, model_name_or_dir, num_labels, dropout=0.1, pretrained=True):
        super().__init__()
        if pretrained:
            self.encoder = AutoModel.from_pretrained(model_name_or_dir)
        else:   # build the architecture only; weights come from a saved state_dict
            self.encoder = AutoModel.from_config(AutoConfig.from_pretrained(model_name_or_dir))
        self.hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.hidden_size, num_labels)

    @staticmethod
    def mean_pool(last_hidden, attention_mask):
        mask = attention_mask.unsqueeze(-1).to(last_hidden.dtype)
        return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

    def forward(self, input_ids, attention_mask, return_embedding=False):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        emb = self.mean_pool(out.last_hidden_state, attention_mask)
        logits = self.classifier(self.dropout(emb))
        return (logits, emb) if return_embedding else logits

model = EmotionTransformer(CFG.model_name, NUM_LABELS, CFG.dropout).to(device)
print("Parameters: %.1fM" % (sum(p.numel() for p in model.parameters()) / 1e6))

## 3. Loss

GoEmotions is heavily imbalanced (see your EDA: `neutral`/`admiration` vs `grief`/`pride`). `bce_posweight` uses `sqrt(neg/pos)` per class, clipped, which boosts rare emotions without the instability of the raw ratio.

In [ ]:
class FocalBCE(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        return (((1 - p_t) ** self.gamma) * bce).mean()

def build_criterion():
    if CFG.loss_type == "bce":
        return nn.BCEWithLogitsLoss()
    if CFG.loss_type == "focal":
        return FocalBCE(CFG.focal_gamma)
    if CFG.loss_type == "bce_posweight":
        y = np.array(tok_ds["train"]["labels"], dtype=np.float32)
        pos = y.sum(0); neg = len(y) - pos
        w = np.clip(np.sqrt(neg / np.maximum(pos, 1)), 1.0, CFG.pos_weight_clip)
        return nn.BCEWithLogitsLoss(pos_weight=torch.tensor(w, dtype=torch.float32, device=device))
    raise ValueError(CFG.loss_type)

criterion = build_criterion()
print("Loss:", CFG.loss_type)

## 4. Training

In [ ]:
no_decay = ("bias", "LayerNorm.weight")
enc_named = list(model.encoder.named_parameters())
optimizer = torch.optim.AdamW([
    {"params": [p for n, p in enc_named if not any(nd in n for nd in no_decay)],
     "lr": CFG.lr_encoder, "weight_decay": CFG.weight_decay},
    {"params": [p for n, p in enc_named if any(nd in n for nd in no_decay)],
     "lr": CFG.lr_encoder, "weight_decay": 0.0},
    {"params": model.classifier.parameters(), "lr": CFG.lr_head, "weight_decay": 0.0},
])

total_steps = len(train_loader) * CFG.epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(CFG.warmup_ratio * total_steps), total_steps
)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
print("Total steps:", total_steps, "| AMP:", use_amp)

In [ ]:
def train_one_epoch():
    model.train()
    total = 0.0
    for batch in train_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device).float()

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(ids, mask)
        loss = criterion(logits.float(), labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += loss.item()
    return total / len(train_loader)

@torch.no_grad()
def collect(loader):
    # returns probabilities, true labels, mean loss
    model.eval()
    probs, labs, total = [], [], 0.0
    for batch in loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device).float()
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(ids, mask)
        total += criterion(logits.float(), labels).item()
        probs.append(torch.sigmoid(logits.float()).cpu().numpy())
        labs.append(labels.cpu().numpy())
    return np.vstack(probs), np.vstack(labs), total / len(loader)

def quick_metrics(labels, preds):
    return {
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "precision_micro": precision_score(labels, preds, average="micro", zero_division=0),
        "recall_micro": recall_score(labels, preds, average="micro", zero_division=0),
        "hamming_loss": hamming_loss(labels, preds),
    }

In [ ]:
history, best_f1, bad_epochs = [], -1.0, 0
ckpt_path = os.path.join(CFG.out_dir, "model.pt")

for epoch in range(1, CFG.epochs + 1):
    t0 = time.time()
    train_loss = train_one_epoch()
    vp, vl, val_loss = collect(val_loader)
    m = quick_metrics(vl, (vp >= CFG.select_threshold).astype(int))
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, **m})

    print(f"Epoch {epoch}/{CFG.epochs} | {time.time()-t0:.0f}s | train {train_loss:.4f} | "
          f"val {val_loss:.4f} | micro-F1 {m['micro_f1']:.4f} | macro-F1 {m['macro_f1']:.4f}")

    if m["macro_f1"] > best_f1:
        best_f1, bad_epochs = m["macro_f1"], 0
        torch.save(model.state_dict(), ckpt_path)
    else:
        bad_epochs += 1
        if bad_epochs >= CFG.patience:
            print("Early stopping.")
            break

history_df = pd.DataFrame(history)
history_df

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
ax[0].plot(history_df["epoch"], history_df["val_loss"], label="validation")
ax[0].set_title("Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(history_df["epoch"], history_df["micro_f1"], label="micro F1")
ax[1].plot(history_df["epoch"], history_df["macro_f1"], label="macro F1")
ax[1].set_title(f"Validation F1 (threshold {CFG.select_threshold})"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. Threshold tuning (validation only)

- One **global** threshold maximising micro-F1.
- One **per-emotion** threshold maximising that emotion's F1.
- Emotions with fewer than `min_support` validation positives (e.g. `grief`, `pride`, `relief`) are too small to tune reliably, so they fall back to the global threshold instead of overfitting to a handful of examples.

In [ ]:
model.load_state_dict(torch.load(ckpt_path, map_location=device))

val_probs, val_labels, _ = collect(val_loader)
test_probs, test_labels, _ = collect(test_loader)

GRID = np.arange(0.05, 0.91, 0.01)

def f1_over_grid(p_col, y_col, grid):
    preds = p_col[:, None] >= grid[None, :]
    pos = (y_col[:, None] == 1)
    tp = (preds & pos).sum(0); fp = (preds & ~pos).sum(0); fn = (~preds & pos).sum(0)
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)

def tune_thresholds(probs, labels, grid=GRID, min_support=20):
    preds = probs[:, :, None] >= grid[None, None, :]
    pos = (labels[:, :, None] == 1)
    tp = (preds & pos).sum((0, 1)); fp = (preds & ~pos).sum((0, 1)); fn = (~preds & pos).sum((0, 1))
    micro = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
    t_global = float(grid[micro.argmax()])

    thr = np.full(probs.shape[1], t_global)
    support = labels.sum(0)
    for c in range(probs.shape[1]):
        if support[c] >= min_support:
            thr[c] = grid[f1_over_grid(probs[:, c], labels[:, c], grid).argmax()]
    return thr, t_global

best_thr, t_global = tune_thresholds(val_probs, val_labels)
print("Global threshold:", round(t_global, 2))

pd.DataFrame({
    "emotion": label_names,
    "val_support": val_labels.sum(0).astype(int),
    "threshold": best_thr.round(2),
})

## 6. Evaluation on the test set

In [ ]:
comparison = pd.DataFrame({
    "fixed_0.5":        quick_metrics(test_labels, (test_probs >= 0.5).astype(int)),
    "global_tuned":     quick_metrics(test_labels, (test_probs >= t_global).astype(int)),
    "per_emotion_tuned": quick_metrics(test_labels, (test_probs >= best_thr).astype(int)),
})
comparison

In [ ]:
preds_tuned = (test_probs >= best_thr).astype(int)
print(classification_report(test_labels, preds_tuned, target_names=label_names, zero_division=0))

f1_df = pd.DataFrame({
    "emotion": label_names,
    "f1": f1_score(test_labels, preds_tuned, average=None, zero_division=0),
    "support": test_labels.sum(0).astype(int),
}).sort_values("f1", ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=f1_df, x="f1", y="emotion")
plt.title(f"Per-emotion F1 on test ({CFG.model_name}, tuned thresholds)")
plt.show()
f1_df

### Ekman-7 view

Most audio/video emotion datasets (e.g. MELD, RAVDESS) use basic emotions, not 28 fine-grained ones. The standard GoEmotions-to-Ekman grouping lets the text branch output labels that line up with the other modalities. A group's probability is the max over its member emotions.

In [ ]:
EKMAN = {
    "anger":    ["anger", "annoyance", "disapproval"],
    "disgust":  ["disgust"],
    "fear":     ["fear", "nervousness"],
    "joy":      ["joy", "amusement", "approval", "excitement", "gratitude", "love",
                 "optimism", "relief", "pride", "admiration", "desire", "caring"],
    "sadness":  ["sadness", "disappointment", "embarrassment", "grief", "remorse"],
    "surprise": ["surprise", "realization", "confusion", "curiosity"],
    "neutral":  ["neutral"],
}
ekman_names = list(EKMAN)
ekman_idx = {g: [label_names.index(l) for l in ls] for g, ls in EKMAN.items()}
assert sorted(i for v in ekman_idx.values() for i in v) == list(range(NUM_LABELS)), "mapping must cover all 28 labels once"

def to_ekman(mat):
    return np.stack([mat[:, idx].max(1) for idx in ekman_idx.values()], axis=1)

val_e_probs, val_e_labels = to_ekman(val_probs), to_ekman(val_labels)
test_e_probs, test_e_labels = to_ekman(test_probs), to_ekman(test_labels)

ekman_thr, ekman_global = tune_thresholds(val_e_probs, val_e_labels)
test_e_preds = (test_e_probs >= ekman_thr).astype(int)

print(classification_report(test_e_labels, test_e_preds, target_names=ekman_names, zero_division=0))

### Error analysis

In [ ]:
errors = (preds_tuned != test_labels).sum(1)
worst = np.argsort(-errors)[:12]

for i in worst:
    true = [label_names[j] for j in np.where(test_labels[i] == 1)[0]]
    pred = [label_names[j] for j in np.where(preds_tuned[i] == 1)[0]]
    print("TEXT:", test_texts[i])
    print("TRUE:", true, "| PRED:", pred)
    print("-" * 80)

## 7. Save artifacts and a reusable predictor

Everything needed for inference (weights, tokenizer, encoder config, thresholds, label order, Ekman map) goes into one folder, so the predictor works without re-downloading the pretrained model.

In [ ]:
tokenizer.save_pretrained(CFG.out_dir)
model.encoder.config.save_pretrained(CFG.out_dir)       # writes config.json (encoder architecture)
# model.pt (best weights) was already written during training

meta = {
    "model_name": CFG.model_name,
    "max_length": CFG.max_length,
    "embedding_dim": model.hidden_size,
    "label_names": label_names,
    "thresholds": [float(t) for t in best_thr],
    "global_threshold": t_global,
    "ekman_map": EKMAN,
    "ekman_thresholds": [float(t) for t in ekman_thr],
}
with open(os.path.join(CFG.out_dir, "emotion_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(sorted(os.listdir(CFG.out_dir)))

In [ ]:
class TextEmotionPredictor:
    def __init__(self, artifact_dir, device=None):
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        with open(os.path.join(artifact_dir, "emotion_meta.json")) as f:
            meta = json.load(f)
        self.labels = meta["label_names"]
        self.thr = np.array(meta["thresholds"])
        cal = meta.get("calibration")      # written by section 12; identity until then
        self.cal_a = np.array(cal["a"]) if cal else np.ones(len(self.labels))
        self.cal_b = np.array(cal["b"]) if cal else np.zeros(len(self.labels))
        self.ekman_thr = np.array(meta["ekman_thresholds"])
        self.max_length = meta["max_length"]
        self.ekman_idx = {g: [self.labels.index(l) for l in ls] for g, ls in meta["ekman_map"].items()}

        self.tokenizer = AutoTokenizer.from_pretrained(artifact_dir)
        self.model = EmotionTransformer(artifact_dir, len(self.labels), pretrained=False)
        self.model.load_state_dict(torch.load(os.path.join(artifact_dir, "model.pt"), map_location=self.device))
        self.model.to(self.device).eval()

    def _ekman(self, probs):
        return np.stack([probs[:, idx].max(1) for idx in self.ekman_idx.values()], axis=1)

    @torch.no_grad()
    def encode(self, texts, batch_size=64):
        # returns (probs [N, 28], embeddings [N, hidden]) - this is what the fusion model uses
        if isinstance(texts, str):
            texts = [texts]
        probs, embs = [], []
        for i in range(0, len(texts), batch_size):
            enc = self.tokenizer(texts[i:i + batch_size], truncation=True, max_length=self.max_length,
                                 padding=True, return_tensors="pt").to(self.device)
            logits, emb = self.model(enc["input_ids"], enc["attention_mask"], return_embedding=True)
            z = np.clip(logits.float().cpu().numpy() * self.cal_a + self.cal_b, -30, 30)
            probs.append(1.0 / (1.0 + np.exp(-z)))
            embs.append(emb.float().cpu().numpy())
        return np.vstack(probs), np.vstack(embs)

    def predict(self, texts, top_k=None):
        single = isinstance(texts, str)
        texts = [texts] if single else list(texts)
        probs, _ = self.encode(texts)
        e_probs = self._ekman(probs)
        out = []
        for i, text in enumerate(texts):
            hits = [(self.labels[j], float(probs[i, j])) for j in range(len(self.labels)) if probs[i, j] >= self.thr[j]]
            hits.sort(key=lambda x: x[1], reverse=True)
            if not hits:                                   # nothing crossed its threshold -> report the top-1
                j = int(probs[i].argmax())
                hits = [(self.labels[j], float(probs[i, j]))]
            ekman = {g: float(e_probs[i, k]) for k, g in enumerate(self.ekman_idx)}
            out.append({
                "text": text,
                "emotions": hits[:top_k] if top_k else hits,
                "ekman": ekman,
                "ekman_top": max(ekman, key=ekman.get),
            })
        return out[0] if single else out

predictor = TextEmotionPredictor(CFG.out_dir)

for s in [
    "I can't believe I finally got the job, thank you so much!",
    "This is so annoying, why does nobody listen to me?",
    "I miss her so much, it still hurts.",
    "Wait, what? You did that without telling anyone?",
]:
    r = predictor.predict(s)
    print(s)
    print("  emotions:", [(e, round(p, 2)) for e, p in r["emotions"]])
    print("  ekman top:", r["ekman_top"], "\n")

## 8. Hooking the text branch into the multimodal system

Per utterance, the text branch exports three things: 28-d emotion probabilities, 7-d Ekman probabilities, and the 768-d embedding. The other branches would export the same kind of vector:

| Branch | Typical encoder | Output |
|---|---|---|
| Text | this notebook (RoBERTa) | 768-d embedding + probabilities |
| Audio | Wav2Vec2 / HuBERT (mean-pooled over time) | 768-d embedding |
| Video | face crops per frame -> CNN/ViT, pooled over frames | 512-d embedding |

`MultimodalEmotionFusion` below is a small gated-attention fusion head. It takes whichever modalities are present (a missing face or silent clip is common in practice) and trains with random modality dropout so it stays robust when one is absent. The audio/video tensors here are random placeholders, only to prove the shapes work.

In [ ]:
def extract_text_features(predictor, texts, path=None):
    probs, emb = predictor.encode(texts)
    feats = {"probs": probs, "ekman": predictor._ekman(probs), "embedding": emb}
    if path:
        np.savez_compressed(path, **feats)
    return feats

demo = extract_text_features(predictor, ["I love this!", "I am so tired of all this."])
print({k: v.shape for k, v in demo.items()})

In [ ]:
class MultimodalEmotionFusion(nn.Module):
    def __init__(self, dims, hidden=256, num_classes=7, dropout=0.3, modality_dropout=0.15):
        super().__init__()
        self.names = list(dims)
        self.modality_dropout = modality_dropout
        self.proj = nn.ModuleDict({
            m: nn.Sequential(nn.Linear(d, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout))
            for m, d in dims.items()
        })
        self.gate = nn.Linear(hidden, 1)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, num_classes)
        )

    def forward(self, feats, present=None):
        # feats: {name: [B, D]}; present: {name: [B] bool} (True if that modality exists for the sample)
        h = torch.stack([self.proj[m](feats[m]) for m in self.names], dim=1)          # [B, M, H]
        B = h.size(0)
        if present is None:
            mask = torch.ones(B, len(self.names), dtype=torch.bool, device=h.device)
        else:
            mask = torch.stack([present[m] for m in self.names], dim=1)

        if self.training and self.modality_dropout > 0:
            keep = mask & (torch.rand(mask.shape, device=h.device) >= self.modality_dropout)
            empty = ~keep.any(dim=1)
            keep[empty] = mask[empty]          # never drop every modality for a sample
            mask = keep

        scores = self.gate(h).squeeze(-1).masked_fill(~mask, float("-inf"))
        w = scores.softmax(dim=1)                                                   # modality weights
        fused = (w.unsqueeze(-1) * h).sum(1)
        return self.classifier(fused), w

fusion = MultimodalEmotionFusion({"text": 768, "audio": 768, "video": 512}).to(device)
B = 4
feats = {"text": torch.randn(B, 768, device=device),
         "audio": torch.randn(B, 768, device=device),
         "video": torch.randn(B, 512, device=device)}
present = {"text": torch.ones(B, dtype=torch.bool, device=device),
           "audio": torch.ones(B, dtype=torch.bool, device=device),
           "video": torch.tensor([True, False, True, False], device=device)}   # two samples have no video

fusion.eval()
logits, w = fusion(feats, present)
print("logits:", tuple(logits.shape), "| modality weights:\n", w.detach().cpu().numpy().round(2))

## 9. Reusable experiment runner

Sections 10 to 15 train several variants, so the training logic is wrapped into helpers here.

- Every run is cached to `artifacts_text/runs/*.npz` (validation and test probabilities). If Colab disconnects, re-run the cell and finished runs are skipped.
- Model selection always uses the **validation** split. The test split is only ever used for reporting.
- Set `TRAIN_SUBSET = 3000` for a fast dry run of sections 9 to 15 (cached under separate names, so it never mixes with real results).

In [ ]:
import copy
from scipy.special import expit as sigmoid

RUN_DIR = os.path.join(CFG.out_dir, "runs")
os.makedirs(RUN_DIR, exist_ok=True)

TRAIN_SUBSET = None     # e.g. 3000 for a quick dry run of sections 9-15
RESULTS = {}            # experiment name -> metrics row, used by the final summary

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def softmax(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

@torch.no_grad()
def predict_logits(net, loader, return_emb=False):
    net.eval()
    L, Y, E = [], [], []
    for batch in loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits, emb = net(ids, mask, return_embedding=True)
        L.append(logits.float().cpu().numpy())
        E.append(emb.float().cpu().numpy())
        Y.append(batch["labels"].numpy())
    L, Y, E = np.vstack(L), np.concatenate(Y, axis=0), np.vstack(E)
    return (L, Y, E) if return_emb else (L, Y)

def predict_probs(net, loader):
    L, Y = predict_logits(net, loader)
    return sigmoid(L), Y

def make_loss(loss_type, y_train):
    if loss_type == "bce":
        return lambda logits, targets: F.binary_cross_entropy_with_logits(logits, targets)
    if loss_type == "focal":
        focal = FocalBCE(CFG.focal_gamma)
        return lambda logits, targets: focal(logits, targets)
    if loss_type == "bce_posweight":
        pos = y_train.sum(0); neg = len(y_train) - pos
        w = np.clip(np.sqrt(neg / np.maximum(pos, 1)), 1.0, CFG.pos_weight_clip)
        pw = torch.tensor(w, dtype=torch.float32, device=device)
        return lambda logits, targets: F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pw)
    raise ValueError(loss_type)

def fit(net, train_loader, loss_fn, score_fn, epochs, lr_enc=None, lr_head=None, patience=None):
    # generic fine-tuning loop: loss_fn(logits, batch) -> loss, score_fn(net) -> float (higher is better)
    lr_enc = lr_enc or CFG.lr_encoder
    lr_head = lr_head or CFG.lr_head
    patience = patience or CFG.patience
    net.to(device)

    no_decay = ("bias", "LayerNorm.weight")
    enc_named = list(net.encoder.named_parameters())
    opt = torch.optim.AdamW([
        {"params": [p for n, p in enc_named if not any(nd in n for nd in no_decay)],
         "lr": lr_enc, "weight_decay": CFG.weight_decay},
        {"params": [p for n, p in enc_named if any(nd in n for nd in no_decay)],
         "lr": lr_enc, "weight_decay": 0.0},
        {"params": net.classifier.parameters(), "lr": lr_head, "weight_decay": 0.0},
    ])
    total = len(train_loader) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(CFG.warmup_ratio * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    best, best_state, bad = -1.0, None, 0
    for ep in range(1, epochs + 1):
        t0 = time.time(); net.train(); run = 0.0
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = net(batch["input_ids"], batch["attention_mask"])
            loss = loss_fn(logits.float(), batch)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item()
        score = score_fn(net)
        print(f"  epoch {ep}/{epochs} | {time.time()-t0:.0f}s | train loss {run/len(train_loader):.4f} | val score {score:.4f}")
        if score > best:
            best = score
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("  early stopping")
                break
    net.load_state_dict(best_state)
    return best

def evaluate_probs(vp, vl, tp, tl, min_support=20):
    # thresholds are tuned on validation only, then applied to test
    thr, tg = tune_thresholds(vp, vl, min_support=min_support)
    m = quick_metrics(tl, (tp >= thr).astype(int))
    m["macro_f1_fixed_0.5"] = f1_score(tl, (tp >= 0.5).astype(int), average="macro", zero_division=0)
    m["val_macro_f1"] = f1_score(vl, (vp >= thr).astype(int), average="macro", zero_division=0)
    return thr, tg, m

def tokenize_dataset(tok):
    def prep(batch):
        enc = tok(batch["text"], truncation=True, max_length=CFG.max_length)
        enc["labels"] = [multi_hot(l) for l in batch["labels"]]
        return enc
    return ds.map(prep, batched=True, remove_columns=ds["train"].column_names)

def run_name(model_name, loss_type, seed, epochs, tag="", subset=None):
    n = f"{model_name.replace('/', '-')}__{loss_type}__seed{seed}__ep{epochs}"
    if tag: n += f"__{tag}"
    if subset: n += f"__sub{subset}"
    return n

def run_experiment(model_name, loss_type, seed, epochs=None, soft_targets=None, alpha=0.5, tag="", label=None):
    epochs = epochs or CFG.epochs
    name = run_name(model_name, loss_type, seed, epochs, tag, TRAIN_SUBSET)
    path = os.path.join(RUN_DIR, name + ".npz")

    if os.path.exists(path):
        z = np.load(path)
        vp, tp = z["val_probs"], z["test_probs"]
        print("cached:", name)
    else:
        print("training:", name)
        set_seed(seed)
        tok = AutoTokenizer.from_pretrained(model_name)
        tds = tokenize_dataset(tok)
        coll = DataCollatorWithPadding(tok)

        train_split = tds["train"]
        if TRAIN_SUBSET:
            train_split = train_split.select(range(TRAIN_SUBSET))
        y_train = np.array(train_split["labels"], dtype=np.float32)
        if soft_targets is not None:        # knowledge distillation: teacher probabilities per training row
            soft = soft_targets[:len(train_split)].astype(np.float32)
            train_split = train_split.add_column("soft", soft.tolist())

        def mk(split, shuffle, bs):
            return DataLoader(split, batch_size=bs, shuffle=shuffle, collate_fn=coll,
                              num_workers=2, pin_memory=use_amp)
        tr_loader = mk(train_split, True, CFG.batch_size)
        va_loader = mk(tds["validation"], False, 128)
        te_loader = mk(tds["test"], False, 128)

        net = EmotionTransformer(model_name, NUM_LABELS, CFG.dropout)
        hard = make_loss(loss_type, y_train)

        def loss_fn(logits, batch):
            l = hard(logits, batch["labels"].float())
            if "soft" in batch:
                l = alpha * l + (1 - alpha) * F.binary_cross_entropy_with_logits(logits, batch["soft"].float())
            return l

        def score_fn(n):
            p, y = predict_probs(n, va_loader)
            return f1_score(y, (p >= CFG.select_threshold).astype(int), average="macro", zero_division=0)

        fit(net, tr_loader, loss_fn, score_fn, epochs)
        vp, _ = predict_probs(net, va_loader)
        tp, _ = predict_probs(net, te_loader)
        np.savez_compressed(path, val_probs=vp, test_probs=tp)
        del net
        torch.cuda.empty_cache()

    thr, tg, m = evaluate_probs(vp, val_labels, tp, test_labels)
    RESULTS[name] = {"experiment": label or name, **m}
    return {"name": name, "val_probs": vp, "test_probs": tp, "thr": thr, "metrics": m}

In [ ]:
# Register the model trained in sections 4-6 as a cached run, so the ablations below reuse it instead of retraining.
main_name = run_name(CFG.model_name, CFG.loss_type, SEED, CFG.epochs, "", None)
main_path = os.path.join(RUN_DIR, main_name + ".npz")
if not os.path.exists(main_path):
    np.savez_compressed(main_path, val_probs=val_probs, test_probs=test_probs)

_, _, m_main = evaluate_probs(val_probs, val_labels, test_probs, test_labels)
RESULTS[main_name] = {"experiment": f"main model ({CFG.model_name}, loss={CFG.loss_type}, seed={SEED})", **m_main}
pd.DataFrame([m_main], index=["main model"])

## 10. Ablations: loss function, model size, seeds

Three questions, answered on the test set but **decided on validation**:

1. Which loss handles the imbalance best: `bce`, `bce_posweight` or `focal`?
2. How much do you lose by using the smaller `distilroberta-base`?
3. How stable is the result across random seeds, and how much does averaging seeds help?

This is 5 new training runs (`bce`, `focal`, distilroberta, and 2 extra seeds); `bce_posweight` is the model you already trained. As a rough guide, expect 10 to 15 minutes per roberta-base run on a T4 (my estimate, not measured). Runs are cached, so you can stop and resume.

In [ ]:
loss_results = {}
for lt in ["bce", "bce_posweight", "focal"]:
    loss_results[lt] = run_experiment(CFG.model_name, lt, SEED, label=f"{CFG.model_name} | loss={lt}")

abl_loss = pd.DataFrame({lt: r["metrics"] for lt, r in loss_results.items()}).T
best_loss = abl_loss["val_macro_f1"].idxmax()      # chosen on VALIDATION, not test
print("Best loss on validation macro-F1:", best_loss)
abl_loss

In [ ]:
small = run_experiment("distilroberta-base", best_loss, SEED, label=f"distilroberta-base | loss={best_loss}")

size_cmp = pd.DataFrame({
    f"roberta-base ({best_loss})": loss_results[best_loss]["metrics"],
    f"distilroberta-base ({best_loss})": small["metrics"],
}).T
size_cmp

In [ ]:
extra_seeds = (SEED + 1, SEED + 2)
seed_runs = [loss_results[best_loss]] + [
    run_experiment(CFG.model_name, best_loss, s, label=f"{CFG.model_name} | loss={best_loss} | seed={s}")
    for s in extra_seeds
]

macro = np.array([r["metrics"]["macro_f1"] for r in seed_runs])
micro = np.array([r["metrics"]["micro_f1"] for r in seed_runs])
print(f"macro-F1 over {len(seed_runs)} seeds: {macro.mean():.4f} +/- {macro.std(ddof=1):.4f}")
print(f"micro-F1 over {len(seed_runs)} seeds: {micro.mean():.4f} +/- {micro.std(ddof=1):.4f}")

# ensemble = average the probabilities of the seeds, then tune thresholds on the averaged validation probabilities
ens_vp = np.mean([r["val_probs"] for r in seed_runs], axis=0)
ens_tp = np.mean([r["test_probs"] for r in seed_runs], axis=0)
ens_thr, ens_tg, ens_m = evaluate_probs(ens_vp, val_labels, ens_tp, test_labels)
RESULTS["ensemble_seeds"] = {"experiment": f"ensemble of {len(seed_runs)} seeds (probability average)", **ens_m}

pd.DataFrame({"single model (seed mean)": {"micro_f1": micro.mean(), "macro_f1": macro.mean()},
              "ensemble": {"micro_f1": ens_m["micro_f1"], "macro_f1": ens_m["macro_f1"]}}).T

In [ ]:
plot_df = pd.DataFrame([
    {"setup": f"loss={lt}", "macro_f1": r["metrics"]["macro_f1"]} for lt, r in loss_results.items()
] + [
    {"setup": "distilroberta-base", "macro_f1": small["metrics"]["macro_f1"]},
    {"setup": f"{len(seed_runs)}-seed mean", "macro_f1": macro.mean()},
    {"setup": f"{len(seed_runs)}-seed ensemble", "macro_f1": ens_m["macro_f1"]},
])
plt.figure(figsize=(8, 4))
sns.barplot(data=plot_df, x="macro_f1", y="setup")
plt.title("Test macro-F1 (per-emotion tuned thresholds)")
plt.xlabel("macro-F1"); plt.ylabel("")
plt.tight_layout(); plt.show()

**How to read this:** the loss ablation tells you whether `bce_posweight` (the default) earns its place. If another loss wins on validation, set `CFG.loss_type` to it and retrain the main model. The seed spread tells you how large a difference has to be before it means anything: gaps smaller than the standard deviation are noise.

## 11. BiLSTM baseline comparison

A fair comparison needs the baseline trained with the same loss, the same threshold tuning and the same splits. This section retrains the BiLSTM with the two fixes identified earlier:

- **Packed sequences**, so the final hidden states are taken at the true end of each sentence, not after the padding.
- **Lower-cased tokens with punctuation split off**, so "great!" and "great" share one word.

If you also want your original notebook's numbers in the table, paste them into `BILSTM_ORIGINAL`.

In [ ]:
import re
from torch.utils.data import TensorDataset
from torch.nn.utils.rnn import pack_padded_sequence

def simple_tok(t):
    return re.findall(r"[a-z0-9']+|[^\sa-z0-9]", t.lower())

class PackedBiLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=200, hid=128, layers=2, dropout=0.3, num_classes=28):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hid, num_layers=layers, batch_first=True, bidirectional=True,
                            dropout=dropout if layers > 1 else 0)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(hid * 2, num_classes)

    def forward(self, ids, lens):
        packed = pack_padded_sequence(self.emb(ids), lens.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        return self.out(self.drop(torch.cat([h[-2], h[-1]], dim=1)))

def run_bilstm(seed=42, epochs=10, max_len=64):
    name = f"bilstm_packed__seed{seed}__ep{epochs}" + (f"__sub{TRAIN_SUBSET}" if TRAIN_SUBSET else "")
    path = os.path.join(RUN_DIR, name + ".npz")

    if os.path.exists(path):
        z = np.load(path); vp, tp = z["val_probs"], z["test_probs"]
        print("cached:", name)
    else:
        print("training:", name)
        set_seed(seed)
        splits = ("train", "validation", "test")
        toks = {s: [simple_tok(t) for t in ds[s]["text"]] for s in splits}
        cnt = Counter(w for t in toks["train"] for w in t)
        bl_vocab = {"<pad>": 0, "<unk>": 1}
        for w, c in cnt.most_common(30000 - 2):
            if c >= 2:
                bl_vocab[w] = len(bl_vocab)

        def encode(s):
            n = len(toks[s])
            ids = np.zeros((n, max_len), dtype=np.int64)
            lens = np.ones(n, dtype=np.int64)
            for i, t in enumerate(toks[s]):
                x = [bl_vocab.get(w, 1) for w in t][:max_len]
                if x:
                    ids[i, :len(x)] = x
                    lens[i] = len(x)
            return torch.tensor(ids), torch.tensor(lens)

        X = {s: encode(s) for s in splits}
        Y = {s: torch.tensor(np.array([multi_hot(l) for l in ds[s]["labels"]], dtype=np.float32)) for s in splits}
        n_tr = TRAIN_SUBSET or len(Y["train"])
        tr = DataLoader(TensorDataset(X["train"][0][:n_tr], X["train"][1][:n_tr], Y["train"][:n_tr]),
                        batch_size=64, shuffle=True)

        net = PackedBiLSTM(len(bl_vocab)).to(device)
        loss_fn = make_loss("bce_posweight", Y["train"][:n_tr].numpy())
        opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-4)

        def probs_of(s):
            net.eval(); out = []
            with torch.no_grad():
                for i in range(0, len(Y[s]), 256):
                    out.append(torch.sigmoid(net(X[s][0][i:i + 256].to(device), X[s][1][i:i + 256])).cpu().numpy())
            return np.vstack(out)

        best, best_state = -1.0, None
        for ep in range(1, epochs + 1):
            net.train()
            for ids, lens, y in tr:
                opt.zero_grad(set_to_none=True)
                loss = loss_fn(net(ids.to(device), lens), y.to(device))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
                opt.step()
            s = f1_score(val_labels, (probs_of("validation") >= CFG.select_threshold).astype(int),
                         average="macro", zero_division=0)
            print(f"  epoch {ep}/{epochs} | val macro-F1 {s:.4f}")
            if s > best:
                best = s
                best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        net.load_state_dict(best_state)
        vp, tp = probs_of("validation"), probs_of("test")
        np.savez_compressed(path, val_probs=vp, test_probs=tp)

    thr, tg, m = evaluate_probs(vp, val_labels, tp, test_labels)
    RESULTS[name] = {"experiment": "BiLSTM (packed, lower-cased tokens)", **m}
    return {"name": name, "val_probs": vp, "test_probs": tp, "thr": thr, "metrics": m}

bilstm = run_bilstm()

In [ ]:
BILSTM_ORIGINAL = None   # optional: paste your original notebook's tuned-threshold metrics as a dict,
                         # using the keys micro_f1, macro_f1, weighted_f1, hamming_loss

cmp_rows = {
    "BiLSTM (packed, lower-cased)": bilstm["metrics"],
    f"{CFG.model_name} (main)": m_main,
}
if BILSTM_ORIGINAL:
    cmp_rows["BiLSTM (your original notebook)"] = BILSTM_ORIGINAL
cols = ["micro_f1", "macro_f1", "weighted_f1", "hamming_loss"]
pd.DataFrame(cmp_rows).T[[c for c in cols if c in pd.DataFrame(cmp_rows).T.columns]]

In [ ]:
def per_class_f1(vp, tp):
    thr, _, _ = evaluate_probs(vp, val_labels, tp, test_labels)
    return f1_score(test_labels, (tp >= thr).astype(int), average=None, zero_division=0)

cmp_df = pd.DataFrame({
    "emotion": label_names,
    "support": test_labels.sum(0).astype(int),
    "BiLSTM": per_class_f1(bilstm["val_probs"], bilstm["test_probs"]),
    "RoBERTa": per_class_f1(val_probs, test_probs),
}).sort_values("support", ascending=False)

long_df = cmp_df.melt(id_vars=["emotion", "support"], value_vars=["BiLSTM", "RoBERTa"],
                      var_name="model", value_name="f1")
plt.figure(figsize=(10, 9))
sns.barplot(data=long_df, x="f1", y="emotion", hue="model", order=cmp_df["emotion"])
plt.title("Per-emotion test F1 (emotions ordered by support, most frequent first)")
plt.tight_layout(); plt.show()

## 12. Probability calibration

The fusion model consumes these probabilities, so they should mean what they say: among comments scored 0.3 for an emotion, about 30% should truly have it. Class-weighted training pushes probabilities **up** for rare emotions, so they are typically over-confident.

The fix is **Platt scaling per emotion**: a slope and intercept on the logit, fitted on the validation split (emotions with fewer than 20 validation positives share one pooled fit). Two things to know:

- It is monotonic per emotion, so the *ranking* of examples does not change. After re-tuning thresholds, F1 should stay about the same. The gain is in probability quality (ECE, Brier score).
- Calibration and thresholds are both fitted on validation, which is fine as long as the test set stays untouched.

In [ ]:
from sklearn.linear_model import LogisticRegression

def to_logits(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def fit_platt(probs, labels, min_support=20):
    z = to_logits(probs); C = probs.shape[1]
    g = LogisticRegression(C=1e4, max_iter=1000).fit(z.reshape(-1, 1), labels.reshape(-1).astype(int))
    a = np.full(C, g.coef_[0, 0]); b = np.full(C, g.intercept_[0])      # pooled fit as the fallback
    for c in range(C):
        if labels[:, c].sum() >= min_support:
            m = LogisticRegression(C=1e4, max_iter=1000).fit(z[:, c:c + 1], labels[:, c].astype(int))
            a[c], b[c] = m.coef_[0, 0], m.intercept_[0]
    return a, b

def apply_platt(probs, a, b):
    return sigmoid(to_logits(probs) * a + b)

def mean_class_ece(probs, labels, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1); eces = []
    for c in range(probs.shape[1]):
        p, y = probs[:, c], labels[:, c]
        idx = np.clip(np.digitize(p, edges) - 1, 0, n_bins - 1)
        e = 0.0
        for k in range(n_bins):
            m = idx == k
            if m.any():
                e += m.mean() * abs(y[m].mean() - p[m].mean())
        eces.append(e)
    return float(np.mean(eces))

a_cal, b_cal = fit_platt(val_probs, val_labels)
cal_val = apply_platt(val_probs, a_cal, b_cal)
cal_test = apply_platt(test_probs, a_cal, b_cal)
cal_thr, cal_tg = tune_thresholds(cal_val, val_labels)

def cal_row(p_test, thr):
    m = quick_metrics(test_labels, (p_test >= thr).astype(int))
    return {"ECE (mean per emotion)": mean_class_ece(p_test, test_labels),
            "Brier score": float(np.mean((p_test - test_labels) ** 2)),
            "macro_f1": m["macro_f1"], "micro_f1": m["micro_f1"]}

cal_table = pd.DataFrame({
    "uncalibrated": cal_row(test_probs, best_thr),
    "Platt-calibrated": cal_row(cal_test, cal_thr),
}).T
cal_table

In [ ]:
def reliability(p, y, n_bins=10):
    p, y = p.ravel(), y.ravel()
    edges = np.unique(np.quantile(p, np.linspace(0, 1, n_bins + 1)))
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, len(edges) - 2)
    xs = [p[idx == k].mean() for k in range(len(edges) - 1) if (idx == k).any()]
    ys = [y[idx == k].mean() for k in range(len(edges) - 1) if (idx == k).any()]
    return np.array(xs), np.array(ys)

fig, ax = plt.subplots(figsize=(5.5, 5))
for name, p in [("uncalibrated", test_probs), ("Platt-calibrated", cal_test)]:
    x, y = reliability(p, test_labels)
    ax.plot(np.clip(x, 1e-4, 1), np.clip(y, 1e-4, 1), marker="o", label=name)
ax.plot([1e-4, 1], [1e-4, 1], "k--", linewidth=1, label="perfect")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("mean predicted probability"); ax.set_ylabel("observed frequency")
ax.set_title("Reliability (all emotions pooled, log-log)")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Save the calibration and the re-tuned thresholds into the artifact folder, and reload the predictor with them.
e_thr_cal, _ = tune_thresholds(to_ekman(cal_val), to_ekman(val_labels))
meta.update({
    "thresholds": [float(t) for t in cal_thr],
    "global_threshold": float(cal_tg),
    "ekman_thresholds": [float(t) for t in e_thr_cal],
    "calibration": {"a": a_cal.tolist(), "b": b_cal.tolist()},
})
with open(os.path.join(CFG.out_dir, "emotion_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

CAL = {"a": a_cal, "b": b_cal, "thr": cal_thr}
predictor = TextEmotionPredictor(CFG.out_dir)       # now applies calibration + calibrated thresholds
print(predictor.predict("I can't believe I finally got the job, thank you so much!"))

## 13. MELD: fine-tuning on dialogue data

GoEmotions is Reddit text. The multimodal system will work on **spoken dialogue**, and MELD (clips from the TV show *Friends*) has text, audio and video for every utterance with 7 emotion labels that match the Ekman grouping from section 6.

This section answers three questions with one table:

1. How well does the GoEmotions model do on MELD **with no training on MELD** (zero-shot, via the Ekman mapping)?
2. Does starting from the GoEmotions model beat starting from plain RoBERTa?
3. Does giving the model the **previous 2 utterances** of the dialogue as context help?

Details that matter:
- Data comes from the official MELD repository CSVs (9,989 train, 1,109 dev, 2,610 test utterances).
- The model is single-label (softmax, cross-entropy with square-root inverse-frequency class weights), because each MELD utterance has exactly one emotion.
- With context, the input is `[previous turns] </s></s> [current utterance]`. If it is too long, the *oldest* context is cut first.
- The embedding that goes to fusion is mean-pooled over the whole input, so with context it also contains a little of the previous turns.
- The main metric is **weighted F1**, the usual MELD metric. Macro-F1 is also reported because `fear` and `disgust` are rare (about 2.7% each of training utterances).
- The best configuration is chosen on the **dev** split.

In [ ]:
MELD_BASE = "https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD"
meld_raw = {s: pd.read_csv(f"{MELD_BASE}/{s}_sent_emo.csv") for s in ("train", "dev", "test")}
MELD_LABELS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
meld_l2i = {l: i for i, l in enumerate(MELD_LABELS)}
MELD_MAX_LEN = 128

def build_meld(df, context_turns):
    df = df.sort_values(["Dialogue_ID", "Utterance_ID"]).reset_index(drop=True)
    ctx = []
    for _, g in df.groupby("Dialogue_ID", sort=False):
        u = g["Utterance"].astype(str).tolist()
        for i in range(len(u)):
            ctx.append(" ".join(u[max(0, i - context_turns):i]) if context_turns > 0 else "")
    out = pd.DataFrame({
        "context": ctx,
        "text": df["Utterance"].astype(str).str.strip(),
        "label": df["Emotion"].map(meld_l2i),
        "dialogue_id": df["Dialogue_ID"],
        "utterance_id": df["Utterance_ID"],
    })
    assert len(ctx) == len(df) and out["label"].notna().all()
    out["label"] = out["label"].astype(int)
    return out

def meld_loaders(tok, context_turns, shuffle_train=True, bs=32):
    from datasets import Dataset
    coll = DataCollatorWithPadding(tok)
    loaders, frames = {}, {}
    for s in ("train", "dev", "test"):
        d = build_meld(meld_raw[s], context_turns)
        frames[s] = d
        hf = Dataset.from_pandas(d[["context", "text", "label"]].rename(columns={"label": "labels"}),
                                 preserve_index=False)
        def enc(b):
            if context_turns > 0:
                return tok(b["context"], b["text"], truncation="only_first", max_length=MELD_MAX_LEN)
            return tok(b["text"], truncation=True, max_length=MELD_MAX_LEN)
        hf = hf.map(enc, batched=True, remove_columns=["context", "text"])
        loaders[s] = DataLoader(hf, batch_size=bs if s == "train" else 128,
                                shuffle=(s == "train" and shuffle_train), collate_fn=coll)
    return loaders, frames

print({s: len(d) for s, d in meld_raw.items()})
print(meld_raw["train"]["Emotion"].value_counts().to_dict())

In [ ]:
def meld_metrics(y, pred):
    return {"accuracy": float((pred == y).mean()),
            "weighted_f1": f1_score(y, pred, average="weighted", zero_division=0),
            "macro_f1": f1_score(y, pred, average="macro", zero_division=0)}

# 1) zero-shot: GoEmotions model -> Ekman-7 -> argmax, no MELD training
perm = [ekman_names.index(l) for l in MELD_LABELS]      # reorder Ekman groups to MELD label order

def zero_shot(split):
    d = build_meld(meld_raw[split], 0)
    probs, _ = predictor.encode(d["text"].tolist())
    e = predictor._ekman(probs)[:, perm]
    return meld_metrics(d["label"].values, e.argmax(1))

zs = {s: zero_shot(s) for s in ("dev", "test")}
zs

In [ ]:
def finetune_meld(init, context_turns, epochs=5, seed=SEED):
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(CFG.model_name, truncation_side="left")   # cut the OLDEST context first
    loaders, frames = meld_loaders(tok, context_turns)

    net = EmotionTransformer(CFG.model_name, NUM_LABELS, CFG.dropout)
    if init == "goemotions":
        net.load_state_dict(torch.load(os.path.join(CFG.out_dir, "model.pt"), map_location="cpu"))
    net.classifier = nn.Linear(net.hidden_size, len(MELD_LABELS))      # new 7-way head

    counts = np.bincount(frames["train"]["label"].values, minlength=len(MELD_LABELS)).astype(np.float64)
    w = np.sqrt(counts.sum() / (len(counts) * counts))
    ce = nn.CrossEntropyLoss(weight=torch.tensor(w / w.mean(), dtype=torch.float32, device=device))
    loss_fn = lambda logits, batch: ce(logits, batch["labels"])

    def score_fn(n):
        L, y = predict_logits(n, loaders["dev"])
        return f1_score(y, L.argmax(1), average="weighted", zero_division=0)

    fit(net, loaders["train"], loss_fn, score_fn, epochs, lr_enc=2e-5, lr_head=1e-3, patience=2)

    res = {}
    for s in ("dev", "test"):
        L, y = predict_logits(net, loaders[s])
        pred = L.argmax(1)
        res[s] = {**meld_metrics(y, pred), "pred": pred, "y": y}
    return net, res

meld_results, best_meld = {}, {"key": None, "dev": -1.0}
for init in ("roberta", "goemotions"):
    for ctx in (0, 2):
        key = f"init={init} | context={ctx}"
        print("\n==", key)
        net, res = finetune_meld(init, ctx)
        meld_results[key] = res
        if res["dev"]["weighted_f1"] > best_meld["dev"]:
            best_meld = {"key": key, "dev": res["dev"]["weighted_f1"], "ctx": ctx,
                         "net": copy.deepcopy(net).cpu()}
        del net
        torch.cuda.empty_cache()
print("\nSelected on dev weighted-F1:", best_meld["key"])

In [ ]:
rows = {"zero-shot (GoEmotions model, no MELD training)": {
    "dev_weighted_f1": zs["dev"]["weighted_f1"], "test_weighted_f1": zs["test"]["weighted_f1"],
    "test_macro_f1": zs["test"]["macro_f1"], "test_accuracy": zs["test"]["accuracy"]}}
for key, r in meld_results.items():
    rows[key + ("   <- selected" if key == best_meld["key"] else "")] = {
        "dev_weighted_f1": r["dev"]["weighted_f1"], "test_weighted_f1": r["test"]["weighted_f1"],
        "test_macro_f1": r["test"]["macro_f1"], "test_accuracy": r["test"]["accuracy"]}
meld_table = pd.DataFrame(rows).T
meld_table

In [ ]:
best_res = meld_results[best_meld["key"]]["test"]
print(classification_report(best_res["y"], best_res["pred"], target_names=MELD_LABELS, zero_division=0))

cm = pd.crosstab(pd.Series(best_res["y"], name="true"), pd.Series(best_res["pred"], name="predicted"))
cm = cm.reindex(index=range(len(MELD_LABELS)), columns=range(len(MELD_LABELS)), fill_value=0)
cm_norm = cm.div(cm.sum(1).clip(lower=1), axis=0)
plt.figure(figsize=(7, 6))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=MELD_LABELS, yticklabels=MELD_LABELS)
plt.title(f"MELD test confusion matrix, row-normalised ({best_meld['key']})")
plt.tight_layout(); plt.show()

### Save the MELD text model and export features for fusion

For every MELD utterance this writes the text embedding (768-d), the 7 class probabilities and the label, keyed by `(dialogue_id, utterance_id)`. The audio and video features must be keyed the same way, because that is what links the three modalities for one utterance.

In [ ]:
MELD_DIR = "artifacts_text_meld"
os.makedirs(MELD_DIR, exist_ok=True)

tok_final = AutoTokenizer.from_pretrained(CFG.model_name, truncation_side="left")
final_net = best_meld["net"].to(device)

torch.save(final_net.state_dict(), os.path.join(MELD_DIR, "model.pt"))
tok_final.save_pretrained(MELD_DIR)
final_net.encoder.config.save_pretrained(MELD_DIR)
with open(os.path.join(MELD_DIR, "meld_meta.json"), "w") as f:
    json.dump({"model_name": CFG.model_name, "labels": MELD_LABELS, "context_turns": best_meld["ctx"],
               "max_length": MELD_MAX_LEN, "truncation_side": "left", "selected": best_meld["key"],
               "embedding_dim": final_net.hidden_size}, f, indent=2)

loaders_final, frames_final = meld_loaders(tok_final, best_meld["ctx"], shuffle_train=False)
for s in ("train", "dev", "test"):
    L, y, E = predict_logits(final_net, loaders_final[s], return_emb=True)
    np.savez_compressed(
        os.path.join(MELD_DIR, f"meld_text_{s}.npz"),
        dialogue_id=frames_final[s]["dialogue_id"].values,
        utterance_id=frames_final[s]["utterance_id"].values,
        embedding=E, probs=softmax(L), labels=y,
    )
    print(s, "embedding", E.shape, "probs", L.shape)
print(sorted(os.listdir(MELD_DIR)))

## 14. Knowledge distillation: a small student for deployment

A smaller model is cheaper to run in a real-time multimodal system. In section 10 `distilroberta-base` was trained directly on the labels. Here it is trained as a **student** of the main model: the loss mixes the hard labels (weight 0.5) with the teacher's predicted probabilities (weight 0.5), so the student also learns *how confident* the teacher is, including which emotions it confuses.

Compare three rows: the teacher, the student trained directly, and the distilled student.

In [ ]:
teacher_loader = make_loader("train", shuffle=False, batch_size=128)      # fixed order so rows line up with the student's data
teacher_train_probs, _ = predict_probs(model, teacher_loader)             # `model` = the main RoBERTa trained earlier
print("teacher probabilities on train:", teacher_train_probs.shape)

kd = run_experiment("distilroberta-base", best_loss, SEED, soft_targets=teacher_train_probs, alpha=0.5, tag="kd",
                    label=f"distilroberta-base distilled from {CFG.model_name}")

kd_table = pd.DataFrame({
    f"teacher: {CFG.model_name}": m_main,
    "student, direct labels only": small["metrics"],
    "student, distilled": kd["metrics"],
}).T[["micro_f1", "macro_f1", "weighted_f1", "hamming_loss"]]
kd_table

## 15. ONNX export and int8 quantization

This exports the **main model** to ONNX and quantizes the weights to int8 (dynamic quantization). It then measures what you give up and what you gain, on the same calibrated thresholds from section 12:

- Output parity between PyTorch and ONNX (should be tiny for fp32).
- Model file size.
- Latency for one sentence on CPU, including tokenization.
- F1 on the first 2,000 test comments, to check that int8 does not hurt accuracy.

Calibration and thresholds are cheap NumPy post-processing, so they are applied outside the ONNX graph. The same `export_onnx` function works for the distilled student if you keep its network.

In [ ]:
!pip install -q onnx onnxruntime

In [ ]:
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

ONNX_DIR = os.path.join(CFG.out_dir, "onnx")
os.makedirs(ONNX_DIR, exist_ok=True)
fp32_path = os.path.join(ONNX_DIR, "model_fp32.onnx")
int8_path = os.path.join(ONNX_DIR, "model_int8.onnx")

class ExportWrapper(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net
    def forward(self, input_ids, attention_mask):
        logits, emb = self.net(input_ids, attention_mask, return_embedding=True)
        return logits, emb

def export_onnx(net, tok, path):
    wrapper = ExportWrapper(copy.deepcopy(net).cpu().float()).eval()
    enc = tok(["I love this!", "Why does nobody ever listen to me?"], padding=True, return_tensors="pt")
    kwargs = dict(
        input_names=["input_ids", "attention_mask"],
        output_names=["logits", "embedding"],
        dynamic_axes={"input_ids": {0: "batch", 1: "seq"}, "attention_mask": {0: "batch", 1: "seq"},
                      "logits": {0: "batch"}, "embedding": {0: "batch"}},
        opset_version=17,
    )
    with torch.no_grad():
        try:
            torch.onnx.export(wrapper, (enc["input_ids"], enc["attention_mask"]), path, dynamo=False, **kwargs)
        except TypeError:       # older torch versions have no `dynamo` argument
            torch.onnx.export(wrapper, (enc["input_ids"], enc["attention_mask"]), path, **kwargs)

export_onnx(model, tokenizer, fp32_path)
quantize_dynamic(fp32_path, int8_path, weight_type=QuantType.QInt8)

sess_fp32 = ort.InferenceSession(fp32_path, providers=["CPUExecutionProvider"])
sess_int8 = ort.InferenceSession(int8_path, providers=["CPUExecutionProvider"])

def ort_logits(sess, texts, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        enc = tokenizer(texts[i:i + bs], padding=True, truncation=True, max_length=CFG.max_length, return_tensors="np")
        out.append(sess.run(["logits"], {"input_ids": enc["input_ids"].astype(np.int64),
                                         "attention_mask": enc["attention_mask"].astype(np.int64)})[0])
    return np.vstack(out)

# parity check: PyTorch vs ONNX fp32 on 64 test comments
enc = tokenizer(test_texts[:64], padding=True, truncation=True, max_length=CFG.max_length, return_tensors="pt")
model.eval()
with torch.no_grad():
    ref = model(enc["input_ids"].to(device), enc["attention_mask"].to(device)).float().cpu().numpy()
print("max |PyTorch - ONNX fp32| on logits:", float(np.abs(ref - ort_logits(sess_fp32, test_texts[:64])).max()))

In [ ]:
N_EVAL = 2000
sub_labels = test_labels[:N_EVAL]

def f1s(probs):
    m = quick_metrics(sub_labels, (probs >= CAL["thr"]).astype(int))
    return m["micro_f1"], m["macro_f1"]

post = lambda z: sigmoid(z * CAL["a"] + CAL["b"])             # calibration applied outside the ONNX graph
p_torch = apply_platt(test_probs[:N_EVAL], CAL["a"], CAL["b"])
p_fp32 = post(ort_logits(sess_fp32, test_texts[:N_EVAL]))
p_int8 = post(ort_logits(sess_int8, test_texts[:N_EVAL]))

cpu_net = copy.deepcopy(model).cpu().float().eval()
def torch_one(t):
    e = tokenizer([t], truncation=True, max_length=CFG.max_length, return_tensors="pt")
    with torch.no_grad():
        cpu_net(e["input_ids"], e["attention_mask"])
def ort_one(sess):
    def f(t):
        e = tokenizer([t], truncation=True, max_length=CFG.max_length, return_tensors="np")
        sess.run(["logits"], {"input_ids": e["input_ids"].astype(np.int64),
                              "attention_mask": e["attention_mask"].astype(np.int64)})
    return f

def latency_ms(fn, texts, warmup=10):
    for t in texts[:warmup]:
        fn(t)
    times = []
    for t in texts[warmup:]:
        s = time.perf_counter(); fn(t); times.append((time.perf_counter() - s) * 1000)
    return float(np.median(times))

bench_texts = test_texts[:110]
torch_size = os.path.getsize(os.path.join(CFG.out_dir, "model.pt")) / 1e6
deploy_table = pd.DataFrame({
    "PyTorch fp32 (CPU)": {"size_mb": torch_size, "latency_ms": latency_ms(torch_one, bench_texts),
                           "micro_f1": f1s(p_torch)[0], "macro_f1": f1s(p_torch)[1]},
    "ONNX fp32":          {"size_mb": os.path.getsize(fp32_path) / 1e6, "latency_ms": latency_ms(ort_one(sess_fp32), bench_texts),
                           "micro_f1": f1s(p_fp32)[0], "macro_f1": f1s(p_fp32)[1]},
    "ONNX int8":          {"size_mb": os.path.getsize(int8_path) / 1e6, "latency_ms": latency_ms(ort_one(sess_int8), bench_texts),
                           "micro_f1": f1s(p_int8)[0], "macro_f1": f1s(p_int8)[1]},
}).T
print(f"F1 measured on the first {N_EVAL} test comments; latency = median per sentence, batch size 1, CPU")
deploy_table

## 16. Final summary

One table with every experiment from this notebook, saved to disk together with a plain-text report.

In [ ]:
summary = pd.DataFrame(list(RESULTS.values()))
keep = [c for c in ["experiment", "micro_f1", "macro_f1", "weighted_f1", "hamming_loss", "macro_f1_fixed_0.5"] if c in summary.columns]
summary = summary[keep].sort_values("macro_f1", ascending=False).reset_index(drop=True)
summary.to_csv(os.path.join(CFG.out_dir, "results_summary.csv"), index=False)

def df_text(df):
    return df.round(4).to_string()

parts = [f"# Text branch report\n\nBase encoder: {CFG.model_name} | dataset: GoEmotions (28 emotions) | thresholds tuned on validation only\n",
         "## GoEmotions experiments (test set)\n" + df_text(summary)]
for title, name in [("Calibration", "cal_table"), ("Distillation", "kd_table"),
                    ("MELD fine-tuning", "meld_table"), ("Deployment", "deploy_table")]:
    obj = globals().get(name)
    if obj is not None:
        parts.append(f"## {title}\n" + df_text(obj))
with open(os.path.join(CFG.out_dir, "text_branch_report.txt"), "w") as f:
    f.write("\n\n".join(parts))

print("Saved:", os.path.join(CFG.out_dir, "results_summary.csv"), "and", os.path.join(CFG.out_dir, "text_branch_report.txt"))
summary

## Where the text pipeline ends

Everything in the text branch is now in place:

| Piece | Where |
|---|---|
| 28-emotion GoEmotions model, tuned thresholds, Ekman-7 view | sections 1 to 7 |
| Ablations (loss, model size, seeds) and ensemble | section 10 |
| Fair BiLSTM comparison | section 11 |
| Calibrated probabilities | section 12 |
| MELD dialogue fine-tuning and fusion-ready features (`artifacts_text_meld/meld_text_*.npz`) | section 13 |
| Distilled student, ONNX and int8 export | sections 14 and 15 |

What is **not** part of the text pipeline, and is the actual next project: the audio branch and the video branch, each producing one embedding per MELD utterance keyed by `(dialogue_id, utterance_id)`, and then training `MultimodalEmotionFusion` (section 8) on the three embeddings together.